# Reading SEC Filings with `edgartools`

In this notebook we will use the [`edgartools`](https://github.com/dgunning/edgartools) Python package to:

1. Look up a public company by its ticker symbol.
2. List its recent **10-K** (annual report) filings.
3. Open the most recent 10-K and inspect its structure.
4. Extract a few named sections — *Business*, *Risk Factors*, and *Management's Discussion and Analysis* — as clean plain text.
5. Save the extracted text to disk so the **next notebook** (`analyze-filings-with-llm.ipynb`) can feed it into an LLM.

We pin `edgartools` to the **5.x** major version so the API calls below behave the same regardless of future major releases.


## 1. Install and import

You only need to run the install cell once per environment. The version pin (`>=5.0.0,<6.0.0`) prevents an accidental jump to a future 6.x release that may rename methods.


In [ ]:
# %pip install --quiet "edgartools>=5.0.0,<6.0.0"


In [ ]:
import os
from pathlib import Path

import edgar
from edgar import Company, set_identity

print("edgartools version:", edgar.__version__ if hasattr(edgar, "__version__") else "(see `pip show edgartools`)")


## 2. Identify yourself to the SEC

The SEC's [fair-access policy](https://www.sec.gov/os/accessing-edgar-data) requires every automated request to include a contact email in the `User-Agent` header. `edgartools` exposes this through `set_identity()`. **Use a real email address you actually monitor** — the SEC may contact you (or rate-limit you) if your scripts misbehave.

We read the identity from an environment variable so we never hard-code a personal email into the notebook. Set `EDGAR_IDENTITY` before launching Jupyter, e.g.:

```bash
export EDGAR_IDENTITY="Jane Student jane@example.edu"
```


In [ ]:
identity = os.environ.get("EDGAR_IDENTITY", "BDI 593 Student student@example.edu")
set_identity(identity)
print("SEC identity set to:", identity)


## 3. Look up a company by ticker

`edgar.Company` accepts either a ticker symbol (e.g. `"AAPL"`) or a CIK (Central Index Key, the SEC's internal company ID). Tickers are easier for humans; CIKs are more stable over time (companies sometimes change tickers).

We will use **Apple Inc.** as our running example. Feel free to change `TICKER` to any large U.S. public company you are curious about.


In [ ]:
TICKER = "AAPL"

company = Company(TICKER)
print(f"Name: {company.name}")
print(f"CIK : {company.cik}")


## 4. List 10-K filings

Every public U.S. company files a **10-K** once per fiscal year. We can ask `edgartools` to filter the company's full filing history down to just 10-Ks.


In [ ]:
tenk_filings = company.get_filings(form="10-K")
tenk_filings.head(5)


The result is a `Filings` collection — think of it as a small DataFrame of metadata (form, filing date, accession number, primary document). Indexing it with `[0]` gives us the most recent filing as a `Filing` object.


In [ ]:
latest_10k = tenk_filings[0]

print("Form           :", latest_10k.form)
print("Filed on       :", latest_10k.filing_date)
print("Period of report:", latest_10k.period_of_report)
print("Accession no.  :", latest_10k.accession_no)
print("Primary doc URL:", latest_10k.homepage_url)


## 5. Parse the 10-K into a structured object

`Filing.obj()` returns a form-aware wrapper. For a 10-K, that wrapper is a `TenK` object that exposes named sections (Business, Risk Factors, MD&A, Financial Statements, …) as attributes. This is much friendlier than parsing the raw HTML yourself.


In [ ]:
tenk = latest_10k.obj()
print("Object type:", type(tenk).__name__)
print("Available items:")
for item in tenk.items:
    print(" -", item)


## 6. Extract specific sections

The three sections most useful for LLM analysis are:

- **Item 1 — Business** (`tenk.business`): a long-form description of how the company makes money.
- **Item 1A — Risk Factors** (`tenk.risk_factors`): the company's own enumeration of risks.
- **Item 7 — Management's Discussion and Analysis** (`tenk.management_discussion`): management's narrative on results.

Each accessor returns a plain Python `str` (already cleaned of HTML).


In [ ]:
business_text = tenk.business or ""
risk_text     = tenk.risk_factors or ""
mdna_text     = tenk.management_discussion or ""

print(f"Business        : {len(business_text):>7,} characters")
print(f"Risk Factors    : {len(risk_text):>7,} characters")
print(f"MD&A            : {len(mdna_text):>7,} characters")


Let's preview the first ~800 characters of each section so we can sanity-check the extraction. A clean extraction should read like English prose, not like leftover HTML markup.


In [ ]:
def preview(label: str, text: str, n: int = 800) -> None:
    print(f"\n=== {label} (first {n} chars) ===\n")
    print(text[:n].strip())
    print("..." if len(text) > n else "")

preview("BUSINESS",     business_text)
preview("RISK FACTORS", risk_text)
preview("MD&A",         mdna_text)


## 7. (Optional) Peek at the financial statements

`edgartools` also parses the XBRL-tagged financial statements into pandas-friendly tables. We will not use these in the LLM notebook, but it is worth knowing they are available — they are useful for combining quantitative trends with the qualitative analysis the LLM will perform.


In [ ]:
try:
    financials = tenk.financials
    income_statement = financials.income_statement()
    print("Income statement shape:", getattr(income_statement, "shape", "n/a"))
    income_statement
except Exception as exc:  # noqa: BLE001
    print("Financials not available for this filing:", exc)


## 8. Save extracted text for the next notebook

We persist each section to `extracted/` so the LLM notebook can read it back without re-querying the SEC. Sticking to plain `.txt` files keeps the data inspectable and avoids any pickle/portability headaches.


In [ ]:
out_dir = Path("extracted")
out_dir.mkdir(exist_ok=True)

meta = {
    "ticker":           TICKER,
    "company_name":     company.name,
    "cik":              str(company.cik),
    "form":             latest_10k.form,
    "accession_no":     latest_10k.accession_no,
    "filing_date":      str(latest_10k.filing_date),
    "period_of_report": str(latest_10k.period_of_report),
}

# Write a small metadata header at the top of each file so the source is always traceable.
def write_section(name: str, text: str) -> Path:
    path = out_dir / f"{TICKER}_{name}.txt"
    header = (
        f"# Source: SEC EDGAR {meta['accession_no']} ({meta['form']})\n"
        f"# Company: {meta['company_name']} (CIK {meta['cik']}, ticker {meta['ticker']})\n"
        f"# Filed:   {meta['filing_date']}  |  Period: {meta['period_of_report']}\n"
        f"# Section: {name}\n\n"
    )
    path.write_text(header + text, encoding="utf-8")
    return path

paths = {
    "business":     write_section("business",     business_text),
    "risk_factors": write_section("risk_factors", risk_text),
    "mdna":         write_section("mdna",         mdna_text),
}

for name, path in paths.items():
    print(f"Wrote {name:<13} -> {path}  ({path.stat().st_size:>7,} bytes)")


## 9. Recap

We have:

- Set the SEC identity required for polite, compliant API access.
- Looked up Apple Inc. by ticker, listed its 10-K filings, and grabbed the most recent one.
- Parsed the 10-K into a `TenK` object and extracted the *Business*, *Risk Factors*, and *MD&A* sections as clean plain text.
- Saved each section to disk under `extracted/` for downstream use.

In the companion notebook, **`analyze-filings-with-llm.ipynb`**, we will load these `.txt` files and run a series of OpenAI-compatible chat-completion calls against them — some with free-form text outputs, some with strict JSON schemas.

:::{tip} Try this on your own
Re-run the notebook with a different `TICKER` (e.g. `"MSFT"`, `"NVDA"`, `"WMT"`) and compare how long each section is across companies. Risk Factor sections in particular vary a lot in length and tone between industries.
:::
